# 13. Jetson Orin Nano / Raspberry Pi 실전 실습 랩

목표: 카메라 모듈이 있는 Jetson 또는 Raspberry Pi에서 각 학습 코스를 실제 실행 흐름으로 따라간다.

이 노트북은 `Basic/work`의 Python 파일을 어떤 순서로 실행하고, 무엇을 성공으로 봐야 하는지 설명합니다.

## Lab 0. 보드 역할 나누기

| 장비 | 잘 맞는 역할 | 주의점 |
|---|---|---|
| Jetson Orin Nano | 카메라, YOLO, ROS2, 경량 로봇 AI, Jetson GPU 실습 | JetPack/Ubuntu/ROS2 버전 호환성 확인 필요 |
| Raspberry Pi | 카메라 캡처, 센서 브릿지, 저수준 제어, ROS2 노드 | 무거운 YOLO/VLM은 느릴 수 있음 |
| PC/노트북 | Gazebo/RViz/학습, 큰 모델 추론 | 실제 센서와 네트워크 연결 필요 |

권장 구조는 `Pi는 센서/제어`, `Jetson은 비전/추론`, `PC는 개발/시뮬레이션`으로 역할을 나누는 것입니다.

## Lab 1. 공통 환경 점검

실행:

```bash
cd Basic/work
python shared/00_check_platform.py
```

성공 기준:

- 보드 모델 또는 OS 정보가 출력된다.
- Jetson이면 `/etc/nv_tegra_release` 또는 `tegrastats` 힌트가 보인다.
- Raspberry Pi면 `libcamera-hello` 또는 `rpicam-hello` 존재 여부를 확인한다.
- ROS2를 쓸 경우 `ROS_DISTRO`와 `ros2` 경로가 확인된다.

실패 원인:

- ROS2 setup 파일을 source하지 않음
- Python 가상환경과 시스템 패키지 경로가 분리됨
- 보드 OS가 공식 카메라 스택과 맞지 않음

## Lab 2. 카메라 확인

### Jetson CSI 카메라

```bash
cd Basic/work
python jetson_orin/01_jetson_camera_preview.py
```

확인할 것:

- GStreamer pipeline이 출력된다.
- preview 창이 열린다.
- 화면이 뒤집혀 있으면 `--flip` 값을 바꿔본다.

### Raspberry Pi Camera Module

```bash
cd Basic/work
python raspberry_pi/01_picamera2_preview.py
```

확인할 것:

- Picamera2가 import된다.
- preview 창이 열린다.
- 안 되면 먼저 `rpicam-hello` 또는 `libcamera-hello`로 카메라 자체를 확인한다.

### USB 카메라 공통

```bash
python shared/01_opencv_camera_preview.py --source 0
```

## Lab 3. 데이터 수집

모방학습이나 비전 모델 학습의 시작은 데이터입니다. 먼저 이미지를 저장하고 metadata를 남기는 습관을 들입니다.

```bash
python shared/02_capture_dataset.py --camera opencv --source 0 --out data/cup_samples --label cup
```

Raspberry Pi Camera Module이면:

```bash
python shared/02_capture_dataset.py --camera picamera2 --out data/pi_samples --label object
```

성공 기준:

- `s`를 누르면 이미지가 저장된다.
- `metadata.jsonl`에 image path, label, timestamp가 기록된다.
- 같은 물체를 조명/거리/각도를 바꿔 50장 이상 모아본다.

## Lab 4. YOLO로 물체 후보 찾기

```bash
python shared/03_yolo_camera_infer.py --source 0 --model yolov8n.pt
```

성공 기준:

- 카메라 프레임 위에 box와 class가 표시된다.
- FPS가 너무 낮으면 더 작은 모델, 낮은 해상도, Jetson TensorRT 변환을 고려한다.

중요한 해석:

YOLO의 bounding box는 `이미지 좌표`입니다. 로봇팔이나 Nav2가 바로 사용할 수 있는 `로봇 좌표`가 아닙니다. 실제 작업에는 camera calibration, depth, TF 변환이 추가로 필요합니다.

## Lab 5. Ollama로 자연어 명령을 구조화하기

```bash
ollama serve
ollama pull llama3.1
python shared/04_ollama_command_parser.py "빨간 컵을 집어 왼쪽 상자에 넣어줘"
```

성공 기준:

- JSON 형태의 `intent`, `objects`, `plan`, `safety_checks`가 출력된다.
- plan의 action이 허용 목록 안에 있다.

중요한 원칙:

LLM 출력은 모터 명령이 아닙니다. `detect_object`, `pick`, `place` 같은 중간 action으로 제한한 뒤, 실제 실행은 ROS2/MoveIt2/Nav2가 담당해야 합니다.

## Lab 6. ROS2 카메라 노드 구조 이해

```bash
python shared/05_ros2_camera_node_guide.py --check
```

이 파일은 실제 패키지를 만들기 전 최소 카메라 노드 구조를 보여줍니다. 다음 단계에서는 ROS2 workspace를 만들고 `sensor_msgs/Image`를 publish하는 패키지로 확장하면 됩니다.

확인할 topic:

```bash
ros2 topic list
ros2 topic hz /camera/image_raw
ros2 run rqt_image_view rqt_image_view
```

## Lab 7. SLAM은 두 경로로 시작한다

### 경로 A: 시뮬레이션으로 먼저 성공하기

SLAM을 처음 배울 때는 TurtleBot3/Gazebo가 가장 재현성이 좋습니다.

```bash
python slam_nav2/02_slam_nav2_command_builder.py --ros-distro humble --mode turtlebot3
```

출력된 명령을 터미널별로 실행합니다.

성공 기준:

- Gazebo에서 로봇이 보인다.
- RViz에서 `/map`, `/scan`, `TF`가 보인다.
- 로봇을 움직이면 occupancy grid가 확장된다.
- `map_saver_cli`로 map 파일이 저장된다.

### 경로 B: 실제 로봇으로 하기

실제 보드에서 2D SLAM을 하려면 최소한 다음이 필요합니다.

- 2D LiDAR가 `/scan`을 publish
- wheel odometry 또는 visual odometry가 `/odom`을 publish
- `base_link`, `laser`, `odom` TF가 연결됨
- 로봇 footprint와 costmap 파라미터가 현실 크기와 맞음

## Lab 8. 카메라만 있을 때의 현실적인 SLAM 전략

카메라 모듈만으로도 Visual SLAM을 공부할 수 있지만, Nav2의 표준 2D 주행 예제와는 다릅니다.

```bash
python slam_nav2/03_camera_only_slam_options.py
```

판단 기준:

- `처음 SLAM`: TurtleBot3 시뮬레이션 또는 2D LiDAR 추천
- `카메라 공부`: OpenCV, calibration, Visual Odometry, ORB-SLAM 개념 추천
- `로봇 주행`: LiDAR/Depth/bumper 등 장애물 source가 costmap에 들어와야 안정적

## Lab 9. 모방학습 데이터 흐름 확인

```bash
python shared/06_imitation_learning_toy.py --data data/demo_actions.jsonl
```

처음에는 실제 학습보다 데이터 schema를 정하는 것이 더 중요합니다.

권장 schema:

```json
{"image":"images/0001.jpg","joint_state":[0.1,0.2],"gripper":0,"instruction":"pick cup","action":[0.0,0.1,1.0]}
```

다음 단계:

- 텔레옵으로 demonstration을 모은다.
- 실패한 장면을 별도로 태깅한다.
- offline validation 후 실제 로봇에는 속도/범위 제한을 걸고 실행한다.

## 최종 MVP 실습 목표

가장 현실적인 첫 MVP는 아래 중 하나입니다.

1. Jetson/RPi 카메라로 컵을 탐지하고, Ollama가 자연어 명령을 JSON plan으로 바꾼다.
2. TurtleBot3 시뮬레이션에서 SLAM 지도를 만들고 Nav2로 목표 이동한다.
3. MoveIt2 시뮬레이션에서 known pose의 물체를 Pick & Place sequence로 옮긴다.
4. OMX 텔레옵 데이터를 20개 이상 저장하고, 관측/action schema를 검증한다.

처음부터 네 가지를 한 번에 묶지 마세요. 하나씩 성공시킨 뒤 ROS2 topic/action/service로 연결해야 디버깅할 수 있습니다.